# Build Android APK with Buildozer on Google Colab

This notebook compiles your Kivy app into an Android APK.

**Instructions:**
1. Runtime → Restart runtime... (if re-running)
2. Runtime → Run all
3. Wait 30-60 minutes
4. Download the APK from the final step


In [ ]:
# Step 1: Install system dependencies and Python packages
import subprocess, sys, os

print("Installing system dependencies...")
subprocess.run(["apt", "update", "-qq"], check=True)
subprocess.run([
    "apt", "install", "-y", "-qq",
    "python3-pip", "python3-dev", "git", "zip", "unzip",
    "openjdk-17-jdk", "libffi-dev", "libssl-dev",
    "wget", "curl", "build-essential", "autoconf", "libtool",
    "pkg-config", "zlib1g-dev", "libbz2-dev"
], check=True)

# Set JAVA_HOME
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

print("\nInstalling Cython 0.29.19...")
subprocess.run([sys.executable, "-m", "pip", "install", "cython==0.29.19"], check=True)

print("\nInstalling buildozer 1.5.0...")
subprocess.run([sys.executable, "-m", "pip", "install", "buildozer==1.5.0"], check=True)

# Verify buildozer is importable
subprocess.run([sys.executable, "-c", "import buildozer; print('buildozer OK:', buildozer.__version__)"], check=True)

print("\n✅ Step 1 complete!")

In [ ]:
# Step 2: Patch buildozer source on disk (skip root check)
# Use pip show to find the package location (more reliable than import)
import os, subprocess, sys

print("Finding buildozer package location...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "show", "buildozer"],
    capture_output=True, text=True, check=True
)
location = None
for line in result.stdout.splitlines():
    if line.startswith("Location:"):
        location = line.split(":", 1)[1].strip()
        break

if not location:
    print("ERROR: Cannot find buildozer location")
    sys.exit(1)

init_path = os.path.join(location, "buildozer", "__init__.py")
print(f"Patching: {init_path}")

with open(init_path, 'r') as f:
    lines = f.readlines()

new_lines = []
skip = False
for line in lines:
    if 'def check_root(self):' in line:
        skip = True
        new_lines.append(line)
        new_lines.append('        """Patched: skip root check"""\n')
        new_lines.append('        return\n')
        continue
    if skip:
        if line.startswith('    def ') or line.startswith('class '):
            skip = False
        else:
            continue
    new_lines.append(line)

with open(init_path, 'w') as f:
    f.writelines(new_lines)

print("✅ Patched buildozer: check_root() disabled")

In [ ]:
# Step 3: Pre-install Android SDK command-line tools
import os, subprocess

sdk_root = os.path.expanduser("~/.buildozer/android/platform/android-sdk")
cmdline_dir = os.path.join(sdk_root, "cmdline-tools")
os.makedirs(cmdline_dir, exist_ok=True)

latest_dir = os.path.join(cmdline_dir, "latest")
if not os.path.exists(latest_dir):
    print("Downloading Android command-line tools...")
    subprocess.run([
        "wget", "-q",
        "https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip",
        "-O", "/tmp/cmdline-tools.zip"
    ], check=True)
    subprocess.run(["unzip", "-q", "-o", "/tmp/cmdline-tools.zip", "-d", cmdline_dir], check=True)
    os.rename(os.path.join(cmdline_dir, "cmdline-tools"), latest_dir)
    print("✅ Command-line tools installed.")
else:
    print("✅ Command-line tools already present.")

sdkmanager = os.path.join(latest_dir, "bin", "sdkmanager")
print(f"sdkmanager exists: {os.path.exists(sdkmanager)}")

In [ ]:
# Step 4: Accept SDK licenses and install build-tools + platform
import os, subprocess

sdk_root = os.path.expanduser("~/.buildozer/android/platform/android-sdk")
sdkmanager = f"{sdk_root}/cmdline-tools/latest/bin/sdkmanager"

os.environ["ANDROID_SDK_ROOT"] = sdk_root
os.environ["ANDROID_HOME"] = sdk_root
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

print("Accepting SDK licenses...")
proc = subprocess.Popen(
    [sdkmanager, "--licenses"],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)
stdout, stderr = proc.communicate(input='y\ny\ny\ny\ny\ny\ny\ny\n')
print("Licenses accepted.")

print("\nInstalling build-tools;30.0.3...")
subprocess.run([sdkmanager, "build-tools;30.0.3"], check=True)

print("Installing platforms;android-30...")
subprocess.run([sdkmanager, "platforms;android-30"], check=True)

print("\n✅ SDK components installed.")

In [ ]:
# Step 5: Clone repo and update buildozer.spec
import os, subprocess, shutil, re

if os.path.exists("network-scanner"):
    print("Removing old directory...")
    shutil.rmtree("network-scanner")

print("Cloning repository...")
subprocess.run(["git", "clone", "https://github.com/Loongood666/network-scanner.git"], check=True)
os.chdir("network-scanner/android")
print(f"Working dir: {os.getcwd()}")

# Update buildozer.spec
with open("buildozer.spec", 'r') as f:
    spec = f.read()

# Ensure android.build_tools is set
if 'android.build_tools' not in spec:
    spec += '\nandroid.build_tools = 30.0.3\n'
else:
    spec = re.sub(r'android\.build_tools\s*=\s*\S+', 'android.build_tools = 30.0.3', spec)

# Ensure android.ndk is set to 28c
if 'android.ndk' not in spec:
    spec += '\nandroid.ndk = 28c\n'
else:
    spec = re.sub(r'android\.ndk\s*=\s*\S+', 'android.ndk = 28c', spec)

with open("buildozer.spec", 'w') as f:
    f.write(spec)

print("\n✅ buildozer.spec updated.")
print("\nFiles in android/:")
subprocess.run(["ls", "-la"])

In [ ]:
# Step 6: Build APK (30-60 minutes)
# Output is written to /tmp/buildozer.log for debugging
import os, subprocess, sys

os.chdir("/content/network-scanner/android")

sdk_root = os.path.expanduser("~/.buildozer/android/platform/android-sdk")
os.environ["ANDROID_SDK_ROOT"] = sdk_root
os.environ["ANDROID_HOME"] = sdk_root
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

print("Starting APK build...")
print("This will take 30-60 minutes. Please be patient.\n")
print("Output is being written to /tmp/buildozer.log\n")

with open("/tmp/buildozer.log", "w") as log_file:
    result = subprocess.run(
        [sys.executable, "-m", "buildozer", "-v", "android", "debug"],
        stdout=log_file,
        stderr=subprocess.STDOUT,
        timeout=None
    )

print(f"Build exited with code: {result.returncode}\n")

# Show last 6000 chars of log
with open("/tmp/buildozer.log", "r", errors="replace") as f:
    content = f.read()
    print("=== LAST 6000 CHARS ===")
    print(content[-6000:])

if result.returncode == 0:
    print("\n✅ Build completed successfully!")
else:
    print(f"\n❌ Build failed with return code {result.returncode}")
    sys.exit(1)

In [ ]:
# Step 7: Download the APK
import os
from google.colab import files

apk_dir = "/content/network-scanner/android/bin"
if os.path.exists(apk_dir):
    apks = [f for f in os.listdir(apk_dir) if f.endswith(".apk")]
    if apks:
        print(f"✅ Found APK: {apks[0]}")
        files.download(os.path.join(apk_dir, apks[0]))
    else:
        print("❌ No APK found in bin/.")
        print("Files in bin/:", os.listdir(apk_dir))
else:
    print("❌ bin/ directory not found.")